In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report

from utils import data_loader_utils
from utils import user_defined_functions as udf


# Same as it shows in Decision_Tree.ipynb
DATA_ROOT = Path("./data")
OP = "OP07"
TRAIN_MACHINE = "M01"
TEST_MACHINES = ["M02", "M03"]

ORIGINAL_FS = 2000
TARGET_FS = 1000
WS = 4096   # the paper also indicates the same window length

P = 20  # AR order
TAU_Q = 0.99    # the threshold of M01 when its ok

In [2]:
def list_h5_files(machine, op, label):
    # label: "good" / "bad"
    folder = DATA_ROOT / machine / op / label
    return sorted(folder.glob("*.h5"))

def load_xyz(file_path):
    arr = data_loader_utils.datafile_read(file_path, False)
    return arr[:, :3]

# downsample to 1khz
def resample_xyz(xyz, fs_in=ORIGINAL_FS, fs_out=TARGET_FS):
    xs = udf.resample_milling_data(xyz[:,0], fs_in, fs_out)
    ys = udf.resample_milling_data(xyz[:,1], fs_in, fs_out)
    zs = udf.resample_milling_data(xyz[:,2], fs_in, fs_out)
    return np.stack([xs, ys, zs], axis=1)

# normalize every window (z-score)
def zscore_per_window(w, eps=1e-8):
    mu = w.mean(axis=0, keepdims=True)
    sd = w.std(axis=0, keepdims=True) + eps
    return (w - mu) / sd

def sliding_windows(xyz: np.ndarray, ws=WS):
    n = len(xyz)
    out = []
    for start in range(0, n - ws + 1, int(float(ws)/2)):
        out.append(xyz[start:start+ws])
    return out

In [3]:
WS  = 4096
HOP = WS // 2

def make_windows(x, ws=WS, hop=HOP):
    # x: (N,3)
    out = []
    for s in range(0, x.shape[0] - ws + 1, hop):
        out.append(x[s:s+ws])
    return out

def windows_for_group(machine, label, op="OP07"):
    files = list_h5_files(machine, op, label)  # your function
    windows = []
    for fp in files:
        xyz = load_xyz(fp)          # your function (N,3)
        # apply same preprocessing you use before AR scoring (resample, etc.) if needed
        windows.extend(make_windows(xyz))
    return windows

groups = {}
for m in ["M01","M02","M03"]:
    for lab in ["good","bad"]:
        groups[(m, lab)] = windows_for_group(m, lab, op="OP07")

In [4]:
def acf_1d(x, max_lag=512):
    x = x.astype(np.float32, copy=False)
    x = x - x.mean()
    denom = (x @ x) + 1e-12
    r = np.correlate(x, x, mode="full")[len(x)-1:len(x)+max_lag]
    return r / denom  # normalized ACF (r[0] = 1)

def group_signature_acf(windows, axis_idx, max_lag=512):
    # average ACF across windows -> vector length (max_lag+1)
    acc = np.zeros(max_lag + 1, dtype=np.float64)
    for w in windows:
        acc += acf_1d(w[:, axis_idx], max_lag=max_lag)
    return acc / max(len(windows), 1)

In [5]:
AXES = ["X","Y","Z"]
MAX_LAG = 512

sign = {}  # sign[(machine,label,axis)] -> vector
for (m, lab), wins in groups.items():
    for a, ax in enumerate(AXES):
        sign[(m, lab, ax)] = group_signature_acf(wins, a, max_lag=MAX_LAG)


In [6]:
def corr_and_cov(vecA, vecB):
    # scalar Pearson correlation + scalar covariance between the two signature vectors
    corr = np.corrcoef(vecA, vecB)[0,1]
    cov  = np.cov(vecA, vecB, bias=False)[0,1]
    return corr, cov


In [7]:
pairs = [
    (("M01","good"), ("M01","good")),
    (("M01","good"), ("M01","bad")),
    (("M01","good"), ("M02","good")),
    (("M01","good"), ("M03","good")),
    (("M03","good"), ("M01","good")),
    (("M01","good"), ("M02","bad")),
    (("M01","good"), ("M03","bad")),
]

rows = []
for (A, B) in pairs:
    for ax in AXES:
        vA = sign[(A[0], A[1], ax)]
        vB = sign[(B[0], B[1], ax)]
        corr, cov = corr_and_cov(vA, vB)
        rows.append({
            "axis": ax,
            "A": f"{A[0]}-{A[1]}",
            "B": f"{B[0]}-{B[1]}",
            "corr(signature)": corr,
            "cov(signature)": cov,
        })

df = pd.DataFrame(rows)
df


,axis,A,B,corr(signature),cov(signature)
0,X,M01-good,M01-good,1.000000,0.068791
1,Y,M01-good,M01-good,1.000000,0.019717
2,Z,M01-good,M01-good,1.000000,0.037427
3,X,M01-good,M01-bad,0.884913,0.049130
4,Y,M01-good,M01-bad,0.659215,0.014135
5,Z,M01-good,M01-bad,0.410994,0.017466
6,X,M01-good,M02-good,0.999534,0.070375
7,Y,M01-good,M02-good,0.710847,0.008939
8,Z,M01-good,M02-good,0.976792,0.030270
9,X,M01-good,M03-good,0.997925,0.059136
